# S14 · Apache Spark y Databricks: del procesamiento distribuido a los modelos
**Universidad Central · Maestría en Analítica de Datos · Big Data 2026-2S**

[Presentación: 24 diapositivas en orden](../Presentaciones/s14-de-dask-a-spark.html#s1) · [Colab](https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/14_PySpark_Compras_Claras.ipynb)

**Pregunta central:** ¿cómo procesar, observar y almacenar datos de contratación pública y entrenar un modelo sobre particiones? Construiremos una consulta verificable, tres capas de calidad y un modelo de práctica. Usamos tres entradas explícitas: 30 ventanas SECOP reales agregadas, seis registros sintéticos con problemas de calidad y 24 textos sintéticos independientes. No son contratos nacionales ni el conjunto evaluado.

**Producto:** resultados conciliados y expediente JSON, tablas o archivos de práctica, evaluación MLlib. La presentación explica cada concepto antes de su uso. Ejecuta en el orden de este cuaderno. **Entorno principal: Databricks Free Edition.** Importa este IPYNB en tu Workspace; el código usa Spark Serverless y persiste tablas Delta. Colab es respaldo local (Parquet, no Delta). No son dos rutas curriculares.

<!-- PEDAGOGIA-RUTA-v1 -->
**Ruta esencial de aprendizaje · cuatro estaciones.** Lee las estaciones antes de ejecutar; amplía únicamente cuando hayas obtenido la evidencia.
<details><summary>Ver recorrido, comprobaciones y experimento formativo</summary>
<ol>
<li><strong>Preparar:</strong> Bloques 1–4: abre Databricks o ruta Colab y carga SECOP. <em>Evidencia:</em> Spark.version y esquema.</li>
<li><strong>Observar:</strong> Bloques 5–10: transforma, ejecuta, compara SQL y joins. <em>Evidencia:</em> Plan, conteos y equivalencia con referencia.</li>
<li><strong>Persistir:</strong> Bloques 11–15: construye Bronze/Silver/Gold y tablas. <em>Evidencia:</em> Tablas Delta o Parquet según entorno.</li>
<li><strong>Entrenar:</strong> Bloque 16: MLlib, pipeline y F1. <em>Evidencia:</em> Predicciones, matriz y limitación.</li>
</ol>
<p><strong>Experimento complementario:</strong> copia este código en una celda nueva cuando corresponda. No se ejecuta automáticamente y no sustituye el laboratorio original.</p>
<pre><code># Ejecutar después de preparar Spark en S14; sin descargas.
from pyspark.sql import functions as F
hechos=spark.range(1000).withColumn(&quot;k&quot;,F.col(&quot;id&quot;)%5)
dim=spark.range(4).select(F.col(&quot;id&quot;).alias(&quot;k&quot;))
a=hechos.join(F.broadcast(dim),&quot;k&quot;,&quot;left&quot;)
b=hechos.join(dim.hint(&quot;merge&quot;),&quot;k&quot;,&quot;left&quot;)
a.explain(); b.explain()
assert a.count()==b.count()==1000
assert not a.exceptAll(b).limit(1).count()
assert not b.exceptAll(a).limit(1).count()</code></pre>
<p><strong>Cómo se lee:</strong> Se comparan resultados y planes; el optimizador puede adaptar la estrategia sugerida.</p>
<p><strong>Modifica:</strong> Repite variando solo dimensión o particiones; registra BroadcastHashJoin/SortMergeJoin si aparecen.</p>
<p><strong>Qué no permite concluir:</strong> Un hint no garantiza el plan; muestra pequeña no prueba ventaja de velocidad.</p>
</details>
<!-- /PEDAGOGIA-RUTA-v1 -->

## Mapa único de aprendizaje y evidencia
| Diapositivas | Bloques de código | Resultado que comprobar |
|---|---|---|
| 1–8 | 1, 8C | Spark disponible y paridad de cálculo |
| 9–13 | 8, 2–7, 8B, 9, 10 | Particiones, esquema, filtro, SQL y plan |
| 14 | 8A | 100.000 filas conciliadas y trabajos observables |
| 15–18 | 11B, 11 | 6/4/2 sintético y 30/30/3 real |
| 19–21 | 12, 12B, 13, 13B | Persistencia e historial si hay Delta |
| 22–24 | 14–17 | Tabla Delta de 24 textos en Databricks, 16 train, 8 test, matriz y F1 |

**HAZ ESTO AHORA.** Abre la presentación en S1. Cada bloque tiene un identificador estable, aunque el orden del cuaderno se reorganizó. Puedes consultar las salidas de una demostración preparada sin repetir cómputo, pero no existe otra secuencia curricular. En local usamos CSV/JSON/Parquet; en Databricks Serverless usamos tablas Delta y Query Profile. Si cambias de kernel, vuelve a ejecutar los bloques previos necesarios. No se requieren tarjetas, tokens ni trabajos recurrentes.

## 1 · Del problema a la herramienta
**pandas:** opera principalmente en la memoria de un proceso; es suficiente si el dato cabe. **Dask:** programa tareas paralelas y flujos similares a pandas. **Apache Spark:** motor con DataFrame API, Spark SQL y planificación distribuida. **Databricks:** plataforma gestionada que ejecuta Spark y añade herramientas de datos, gobernanza y automatización.

En un despliegue distribuido Spark separa **driver** (coordina), **executors** (procesan) y **particiones** (porciones lógicas del dato). Un **job** nace de una acción; un **stage** agrupa tareas; una **task** procesa una partición. **Shuffle** es el intercambio de filas entre particiones, típico al agrupar o unir por claves.

**No confundir:** usar `local[2]` en Colab NO significa dos servidores. Una partición no equivale a un worker. En Databricks serverless parte de la infraestructura está administrada y no siempre se expone la Spark UI clásica.

**Ejemplo de arquitectura:** cuatro particiones y dos ranuras de ejecución permiten dos tareas simultáneas, con otras dos esperando; no representan cuatro computadores. `groupBy` suma resultados parciales y puede redistribuir por clave. `local[2]` significa dos hilos de ejecución de Spark en la misma máquina. Spark Connect en Databricks envía planes al servicio; no proporciona acceso a RDD ni a SparkContext clásico. El Query Profile muestra información de consultas, no necesariamente número de máquinas.

### Por qué existe Spark: del disco al plan
Una máquina puede crecer verticalmente (RAM/CPU), pero el crecimiento horizontal añade nodos y exige repartir trabajo, comunicar resultados y recuperarse de fallos. MapReduce (2004) separa map y reduce; Hadoop/HDFS almacena bloques replicados y los trabajos encadenados suelen materializar resultados en disco. Spark nació en Berkeley (2009) para permitir planes de varias operaciones y reutilización de datos; también usa disco, red y memoria. No es sinónimo de Hadoop ni requiere HDFS.

| Decisión | pandas | Dask | Spark |
|---|---|---|---|
| Datos que caben en RAM | Sencillo y eficiente | Sobrecoste posible | Sobrecoste posible |
| Tareas Python independientes | Local | Scheduler y workers flexibles | Modelo de ejecución Spark |
| ETL estructurado | Operaciones locales | DataFrame particionado | Spark SQL y Catalyst |
| ML | sklearn local | Dask-ML o experimentos coordinados | Algoritmos MLlib distribuibles |
| Observabilidad | Medición local | Dashboard | Spark UI / Query Profile |

**En el taller:** 810 filas caben en pandas; usamos Spark para practicar una arquitectura escalable y no para asegurar aceleración. Separar capacidad de la plataforma y necesidad del caso.

### Arquitectura y vocabulario
Spark Core planifica trabajos; Spark SQL ejecuta consultas estructuradas; MLlib aprende modelos; Structured Streaming procesa entradas incrementales y GraphX ofrece grafos en Scala. Un RDD es una colección particionada con linaje para recomputar particiones; un DataFrame tiene esquema y operaciones estructuradas que Catalyst puede optimizar. SparkSession es la puerta de entrada a SQL/DataFrames; SparkContext clásico conecta con los recursos. Driver coordina; Cluster Manager asigna recursos; worker node es una máquina; executor es un proceso; core es un recurso de CPU; ranura (slot) es capacidad de ejecutar una tarea simultáneamente. Un worker puede alojar más de un executor según despliegue. En modo local no hay clúster remoto. En serverless Spark Connect envía planes al servicio, sin acceso al SparkContext clásico ni RDD.

**Pregunta de control:** 12 particiones y 4 ranuras de igual duración requieren idealmente 3 rondas; no son 12 máquinas. El coste real incluye inicialización, comunicación y datos desbalanceados.

Referencias: [Spark cluster](https://spark.apache.org/docs/4.0.1/cluster-overview.html), [RDD paper Berkeley](https://www.usenix.org/conference/nsdi12/technical-sessions/presentation/zaharia), [Stanford CS246](https://web.stanford.edu/class/cs246/).

### Construir el concepto antes de nombrarlo
En S3 cambia 40 GB de datos frente a 16 GB de RAM: el excedente teórico es 24 GB, pero el tamaño en RAM puede ser mayor al archivo. S4 compara aumentar recursos en una máquina (vertical) con agregar nodos (horizontal). S5 compara trabajos MapReduce encadenados con un plan Spark; el motor no elimina por completo escrituras en disco. S6 calcula A=4, B=4, C=4 sobre doce datos; S7 decide según necesidad, no una tabla de velocidades. S8 separa Spark Core, APIs y almacenamiento. En S9 añade driver, gestor, worker, executor y tasks. El reparto real se observa después en código, no se infiere de la animación.

### Por qué Dask y Spark se ven iguales en S6
Ambos pueden particionar, construir grafos, evaluar de forma diferida, optimizar consultas y combinar parciales. El resultado A=4, B=4, C=4 no distingue motores. En Dask DataFrame el optimizador simplifica expresiones y el scheduler coordina operaciones, normalmente sobre pandas/NumPy; en Spark las expresiones nativas PySpark/SQL se convierten en planes que Catalyst optimiza y el motor JVM ejecuta. JVM = máquina virtual Java; las UDF Python pueden ejecutar código Python aparte. Dask también puede reducir shuffle y hacer broadcast. La diferencia es el ecosistema, APIs y estrategias de ejecución, no que uno distribuya y el otro no. [Optimizador Dask](https://docs.dask.org/en/stable/dataframe-optimizer.html) · [Optimización Spark SQL](https://spark.apache.org/docs/4.0.1/sql-performance-tuning.html).

### Qué es realmente un executor (S7–S11)
Una máquina física o virtual aporta CPU, RAM y disco; un proceso es un programa en ejecución con recursos propios; los hilos son flujos de trabajo dentro de un proceso. En Spark clásico el driver coordina la aplicación y solicita recursos al gestor (YARN, Kubernetes o Standalone). El gestor facilita iniciar executors en nodos worker. Un executor es un proceso de la aplicación (normalmente JVM) que recibe tareas, ejecuta operaciones, guarda intermedios en memoria/disco y reporta resultados/estado. Puede reutilizarse para múltiples tareas y finalizar con la aplicación o con asignación dinámica. El driver no recibe todas las filas para coordinar. Si se pierde una tarea o un intermedio, Spark puede reintentar/recomputar usando linaje, sujeto a límites y disponibilidad de datos. Una tarea trabaja sobre una partición dentro de una etapa.

**Simula S9:** 2 nodos × 2 executors × 2 CPU = 8 ranuras ideales con una CPU/tarea. 16 particiones requieren 2 rondas ideales. Cada nodo dispone de 12 CPU y 32 GB ilustrativos y cada executor 4 GB. Cambia particiones a 3 y observa recursos libres; no confundas más particiones con más servidores. SparkSession ofrece DataFrames y SQL; SparkContext clásico conecta aplicación y recursos. En Spark Connect/Serverless no se administra el clúster ni se debe intentar acceder a SparkContext. [Arquitectura](https://spark.apache.org/docs/4.0.1/cluster-overview.html).

## 2 · Antes de ejecutar: preparar Databricks Free Edition
Databricks es una plataforma de ingeniería, análisis y aprendizaje automático en la nube. Spark es el motor que ejecuta operaciones; Workspace organiza código, Serverless asigna cómputo sin administrar nodos, Environment contiene dependencias, Unity Catalog organiza permisos y activos, Delta Lake conserva tablas transaccionales. SQL Editor consulta tablas, Query Profile observa operaciones, Lakeflow Jobs coordina tareas, MLflow registra experimentos. Los datos persisten independientemente de la sesión del notebook.

1. Abre [Databricks Free Edition](https://www.databricks.com/learn/free-edition), crea cuenta o inicia sesión y entra al Workspace. No elijas una prueba de pago por obligación.
2. Abre [este cuaderno en GitHub](https://github.com/jazaineam1/BigData2026/blob/main/Cuadernos/14_PySpark_Compras_Claras.ipynb), pulsa Download raw file y guarda el .ipynb. En Workspace entra a tu carpeta, menú ⋮ → Import → File → selecciona el archivo. [Guía oficial](https://docs.databricks.com/aws/en/notebooks/notebook-export-import).
3. Abre el notebook importado; selecciona Serverless en el selector de cómputo. En el panel Environment, selecciona un Base environment con versión 4 o posterior compatible con pyspark.ml, aplica cambios si la interfaz lo requiere. [Configuración oficial](https://docs.databricks.com/aws/en/compute/serverless/dependencies) · [Environment 4](https://docs.databricks.com/aws/en/release-notes/serverless/environment-version/four). No instales PySpark.
4. En Catalog identifica un catálogo y esquema propios con permisos USE CATALOG, USE SCHEMA y CREATE TABLE. El bloque 1 imprime el contexto. Si el contexto no es adecuado, cambia CATALOGO_S14 y ESQUEMA_S14 por los nombres que ves en Catalog, ejecuta el bloque de nuevo. Los campos vacíos conservan el contexto actual.
5. Ejecuta celda por celda en el orden del cuaderno. Todos los datos de práctica se incluyen en código; no es necesario acceder a la API SECOP desde Databricks ni descargar desde la red. Free Edition puede restringir la red y el cómputo.

**Colab como respaldo:** abre el enlace, Entorno de ejecución → CPU, ejecuta bloque 1; se instala PySpark solo si falta. El mismo código se ejecuta localmente con Parquet sin historial Delta. `local[2]` significa dos hilos locales, no dos nodos.

**Comprobación:** versión Spark/Python, entorno y catálogo/esquema. La prueba de permisos real es la escritura de 12/12B/15; no confundas el catálogo visible con permiso de creación. [Límites Free Edition](https://docs.databricks.com/aws/en/getting-started/free-edition-limitations).

**Transición de clase:** en S8, antes de la primera ejecución, abre Workspace e importa el IPYNB. En el selector de cómputo conecta Serverless, abre Environment a la derecha y elige una versión compatible con MLlib. Ejecuta bloque 1 (Shift+Enter) y conserva el mismo notebook para toda la clase. Regresa a S9. S20 profundiza Catalog y Query Profile, no exige iniciar otro entorno. No instales PySpark en Serverless.

In [ ]:
# 1 · Verificar el entorno y crear una SparkSession de forma repetible
import importlib.util, subprocess, sys, platform
if "spark" not in globals() and importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.1"])
from pyspark.sql import SparkSession, functions as F, types as T
if "spark" not in globals():
    spark = SparkSession.builder.master("local[2]").appName("UC_S14_SECOP").getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    ENTORNO = "Colab / Spark local"
else:
    ENTORNO = "Databricks Serverless" if "dbutils" in globals() else "Spark preexistente"
print("Entorno:", ENTORNO, "| Spark:", spark.version, "| Python:", platform.python_version())
# Databricks: deja vacíos para utilizar tu contexto actual, o cambia AMBOS nombres.
CATALOGO_S14 = ""
ESQUEMA_S14 = ""
if "dbutils" in globals():
    if bool(CATALOGO_S14) != bool(ESQUEMA_S14):
        raise ValueError("Indica ambos nombres o deja ambos vacíos.")
    if CATALOGO_S14:
        spark.sql("USE CATALOG `"+CATALOGO_S14.replace("`","``")+"`")
        spark.sql("USE SCHEMA `"+ESQUEMA_S14.replace("`","``")+"`")
    spark.sql("SELECT current_catalog() AS catalogo, current_schema() AS esquema").show(truncate=False)
    print("Confirma en Catalog que puedes crear tablas en este esquema.")

**Cómo se lee:** `ENTORNO` y `spark.version` indican el motor disponible. **Qué nos dice:** la API PySpark puede ejecutarse. **Qué NO permite concluir todavía:** que existan varios computadores ni cuál es la velocidad de un clúster. **Qué error común:** llamar distribuido a un modo local por tener dos hilos. Si aparece un error de Java en Colab, reinicia el runtime y ejecuta desde el inicio; si falla la cuota de Databricks, sigue por Colab.

### 8C · Comparación de motores (diapositiva 6)
Mismo dato sintético, mismo filtro y agregación, mismos conteos. No compara lectura de archivos ni representa un benchmark de clúster. Cambia `N_COMPARA` entre 1000, 10000 y 100000. Dask solo se ejecuta si ya está instalado; no descarga paquetes. Mide por separado la operación de Spark desde la construcción hasta collect, y en Dask incluye el inicio del cliente. No se puede atribuir diferencias exclusivamente al motor.

**Misma pregunta:** filtrar IDs pares, contar por id % 10. El resultado de 10.000 filas sintéticas es cinco claves con 1.000 registros cada una. `groupby().size()` calcula conteos en pandas; en Dask `compute()` ejecuta el grafo; en Spark `collect()` solicita cinco filas. Dask se compara solo si ya está disponible: no instalar por obligación en Spark Connect. Los tiempos incluyen límites distintos, así que la comparación es de corrección.

**Lee los planes, no solo los tiempos:** Spark explain("formatted") muestra operadores físicos (filtro, agregación, Exchange si corresponde). Dask pprint() muestra la expresión y optimize().pprint() muestra la expresión optimizada; explain() puede dibujar un grafo si Graphviz está disponible. No se necesita instalar Graphviz. Busca filtro, agrupación y combinación. El número de operadores no es un indicador comparable de velocidad.

In [ ]:
# 8C · Paridad pandas, Dask (si disponible) y Spark
import importlib.util, time
import pandas as pd
N_COMPARA=10000
assert N_COMPARA in (1000,10000,100000)
p_compara=pd.DataFrame({"id":range(N_COMPARA)})
p_compara["categoria"]=p_compara.id%10
t=time.perf_counter()
ref_compara={int(k):int(v) for k,v in p_compara[p_compara.id%2==0].groupby("categoria").size().items()}
print("pandas segundos operación:",time.perf_counter()-t)
s_compara=spark.createDataFrame(p_compara)
t=time.perf_counter()
plan_compara=s_compara.filter(F.col("id")%2==0).groupBy("categoria").count()
res_compara={int(r["categoria"]):int(r["count"]) for r in plan_compara.collect()}
assert res_compara==ref_compara
print("Spark segundos operación:",time.perf_counter()-t)
print("Plan físico Spark:")
plan_compara.explain(mode="formatted")
if importlib.util.find_spec("dask") and importlib.util.find_spec("distributed"):
    import dask.dataframe as dd
    from distributed import Client
    t=time.perf_counter()
    with Client(processes=False,n_workers=2,threads_per_worker=1,dashboard_address=None):
        d_compara=dd.from_pandas(p_compara,npartitions=4)
        plan_dask=d_compara[d_compara.id%2==0].groupby("categoria").size()
        d_res={int(k):int(v) for k,v in plan_dask.compute().items()}
    assert d_res==ref_compara
    print("Dask segundos con inicio cliente:",time.perf_counter()-t)
    if hasattr(plan_dask,"pprint") and hasattr(plan_dask,"optimize"):
        print("Expresión Dask antes y después de optimizar:")
        plan_dask.pprint()
        plan_dask.optimize().pprint()
    else:
        print("La versión de Dask no expone pprint/optimize; no se comparó el plan.")
else:
    print("Dask no disponible: no se ejecutó ni se instaló.")
print("Paridad de motores ejecutados:",ref_compara)

**Cómo se lee:** cinco categorías pares, cada una N/10 registros. **Qué nos dice:** paridad del cálculo en los motores ejecutados. **Qué NO permite concluir:** que el más rápido en esta prueba escale mejor; preparación y arranque no son idénticos. **Qué error común:** medir la creación diferida del plan sin ejecutar una acción.

## Particiones, tareas y rondas (diapositivas 9–12)
Antes de usar repartition, abre S9 y construye la aplicación; S10 divide 12 identificadores en tres porciones, S11 ejecuta seis tareas en rondas y S12 reúne claves A/B/C. **Partición**: porción lógica de datos; **task**: cálculo sobre una partición en una etapa; **ranura**: capacidad para una tarea simultánea. Un **shuffle** reúne datos o parciales por clave y puede usar red y disco. El motor puede agrupar parcialmente antes de redistribuir. Aquí comprobamos 40 filas antes/después, no el número de máquinas. `spark.range(inicio,fin,paso,numPartitions)` crea datos sintéticos; `spark_partition_id()` devuelve ID de partición observada; `repartition(n, clave)` solicita redistribución.

### De CPU a capacidad simultánea: ¿qué es una ranura?
Un puesto de trabajo atiende una tarea y se libera cuando termina. En Spark clásico 4.0, si un executor tiene C núcleos asignados y cada tarea requiere T núcleos (spark.task.cpus, por defecto 1), la capacidad teórica es floor(C/T) tareas simultáneas, si no hay otros límites. Para varios executors, suma las capacidades individuales; no agrupes núcleos de máquinas distintas para una tarea. Ejemplo: 4 CPU y 1 por tarea = 4 ranuras; 4 CPU y 2 por tarea = 2 ranuras. Con 6 tareas y 2 ranuras, 3 rondas ideales; con 3 ranuras, 2. Si C<T, no hay capacidad. No son núcleos físicos garantizados, servidores ni particiones; local[2] solicita dos hilos de trabajo locales. En serverless no podemos inferir la capacidad real sin métricas/configuración expuestas. [Spark configuration](https://spark.apache.org/docs/4.0.1/configuration.html).

**S9–11:** la capacidad total es la suma de floor(C/T) de cada executor; si una tarea necesita dos CPU, no puede sumar CPU de dos procesos. Particiones de DataFrame son porciones lógicas, no necesariamente archivos; el linaje permite reconstrucción si la fuente permanece disponible. La prueba de 40 filas muestra IDs reales observados, no cantidad de executors.

In [ ]:
# 8 · Ver particiones sin usar API RDD (compatible con serverless)
base = spark.range(0, 40, 1, 4)
print("ANTES:")
base.withColumn("p", F.spark_partition_id()).groupBy("p").count().orderBy("p").show()
redistribuido = base.repartition(2, (F.col("id") % 3))
print("DESPUÉS:")
redistribuido.withColumn("p", F.spark_partition_id()).groupBy("p").count().orderBy("p").show()
assert redistribuido.count() == 40
print("Filas concretas por partición (solo 40 filas de práctica):")
redistribuido.select("id",F.spark_partition_id().alias("particion")).orderBy("particion","id").show(40)

**Cómo se lee:** el reparto muestra filas por ID de partición y mantiene 40 registros. **Qué nos dice:** las particiones pueden cambiar con una redistribución. **Qué NO permite concluir todavía:** que se usaron dos servidores, que se redujo el tiempo o que el reparto sea uniformemente balanceado. **Qué error común:** llamar *workers* a las particiones.

## 3 · Datos públicos, transformaciones trazables
Fuente: `Datos/tc1_ventanas_resumen.json` (generado por `utils/tc1_data_gates.py` el 2026-10-02T06:22:55+00:00), a partir de SECOP II Procesos (`p6dx-8zbt`) y Contratos (`jbjy-vk9h`) de datos.gov.co. La selección original usa hasta 3.000 procesos y 4.000 contratos por ventana mensual con criterios explícitos: **no es un censo nacional**.

Usamos 30 ventanas con cinco campos: `mes`, `procesos_unicos`, `emparejados`, `cobertura`, `departamento_particion`. Los 30 registros quedan incorporados en el notebook para **evitar 30 descargas simultáneas a Socrata**.

**Lectura manual:** la ventana 2024-01 tiene 3.000 procesos únicos y 201 emparejamientos: `201/3000=0.067` (6,7 %). Esta razón se refiere únicamente al recorte registrado; no demuestra irregularidades de contratación.

In [ ]:
# 2 · Snapshot offline derivado del manifiesto real; no hay consultas a la API
FILAS = [["2024-01",3000,201,0.067,"Cauca"],["2024-02",2945,120,0.0407,"Distrito Capital de Bogotá"],["2024-03",2872,188,0.0655,"Tolima"],["2024-04",2874,130,0.0452,"Distrito Capital de Bogotá"],["2024-05",2858,275,0.0962,"Bolívar"],["2024-06",2920,86,0.0295,"Distrito Capital de Bogotá"],["2024-07",2910,69,0.0237,"Antioquia"],["2024-08",2900,231,0.0797,"Boyacá"],["2024-09",2932,88,0.03,"Distrito Capital de Bogotá"],["2024-10",2924,148,0.0506,"Distrito Capital de Bogotá"],["2024-11",2967,310,0.1045,"Distrito Capital de Bogotá"],["2024-12",2946,169,0.0574,"Distrito Capital de Bogotá"],["2025-01",2990,204,0.0682,"Cauca"],["2025-02",2984,84,0.0282,"Antioquia"],["2025-03",2971,132,0.0444,"Chocó"],["2025-04",2934,147,0.0501,"Antioquia"],["2025-05",2952,101,0.0342,"Distrito Capital de Bogotá"],["2025-06",2915,120,0.0412,"Antioquia"],["2025-07",2946,113,0.0384,"Antioquia"],["2025-08",2872,118,0.0411,"Antioquia"],["2025-09",2902,171,0.0589,"Distrito Capital de Bogotá"],["2025-10",2805,131,0.0467,"Distrito Capital de Bogotá"],["2025-11",2981,138,0.0463,"Distrito Capital de Bogotá"],["2025-12",2879,90,0.0313,"Distrito Capital de Bogotá"],["2026-01",2999,282,0.094,"Boyacá"],["2026-02",2992,70,0.0234,"Distrito Capital de Bogotá"],["2026-03",2902,243,0.0837,"Norte de Santander"],["2026-04",2802,100,0.0357,"Distrito Capital de Bogotá"],["2026-05",2922,85,0.0291,"Distrito Capital de Bogotá"],["2026-06",2766,201,0.0727,"Norte de Santander"]]
PROCEDENCIA = {"fuente":"SECOP II p6dx-8zbt y jbjy-vk9h", "archivo":"Datos/tc1_ventanas_resumen.json", "generado_utc":"2026-10-02T06:22:55+00:00", "script":"utils/tc1_data_gates.py", "grano":"ventana de extracción, no contrato"}
assert len(FILAS) == 30
assert FILAS[0][:3] == ["2024-01", 3000, 201]
print("Ventanas:", len(FILAS), "| Primer registro:", FILAS[0])

## 4 · Tipos: no dejemos que Spark adivine el significado
Una columna llamada `mes` contiene claves `YYYY-MM`, no fechas de contratos; `cobertura` es un número entre 0 y 1, no un valor monetario. Crear el **schema** evita inferencias accidentales.

**Mini ficha `spark.createDataFrame(data, schema=...)`:** entrada: filas y tipos; salida: DataFrame. `printSchema()` muestra los tipos; `show(n)` materializa una vista de n filas. Error típico: dar por hecho que `show(5)` ha explorado toda la fuente.

**Ficha:** `StructType` describe la tabla, `StructField(nombre,tipo,nullable)` define cada columna, `createDataFrame` construye un DataFrame, `printSchema` muestra tipos y `show(5)` devuelve una vista de cinco filas. `count()` comprueba todo el conjunto. El campo cobertura es una razón, no moneda. Ejemplo manual: 201/3000=6,7 %.

In [ ]:
# 3 · Esquema estricto y primera acción
schema = T.StructType([
    T.StructField("mes", T.StringType(), False),
    T.StructField("procesos_unicos", T.LongType(), False),
    T.StructField("emparejados", T.LongType(), False),
    T.StructField("cobertura", T.DoubleType(), False),
    T.StructField("departamento_particion", T.StringType(), True)
])
df = spark.createDataFrame([tuple(f) for f in FILAS], schema=schema)
df.printSchema()
df.orderBy("mes").show(5, truncate=False)
assert df.count() == 30

**Cómo se lee:** se ven tipos y cinco primeras ventanas ordenadas. **Qué nos dice:** hay 30 ventanas con campos definidos. **Qué NO permite concluir todavía:** que la extracción original cubrió la totalidad de SECOP o que la cobertura corresponde al valor del gasto. **Qué error común:** interpretar la columna de departamento como la ubicación de todos los contratos de la ventana; es una partición de control del manifiesto.

## 5 · Transformaciones frente a acciones
`filter`, `select` y `withColumn` normalmente construyen un **plan diferido** (*lazy*). `count`, `show` y escribir resultados activan un trabajo. El optimizador **Catalyst** utiliza el plan para elegir operaciones. Que el código tenga menos líneas no significa que vaya a ejecutarse más rápido.

**Mini fichas:** `filter(condición)` conserva filas; `withColumn(nombre, expresión)` deriva un campo; `count()` devuelve un entero tras ejecutar. Error común: medir solo el tiempo de construir el DataFrame y compararlo con una acción de otro motor.

**DAG (grafo dirigido acíclico):** expresa dependencias entre operaciones; no hay ciclos dentro del grafo de una ejecución. Plan lógico = qué; plan optimizado = simplificaciones de Catalyst; plan físico = cómo. Una acción puede generar uno o más jobs; un job tiene stages separados por dependencias como shuffle; cada stage ejecuta tasks sobre particiones. AQE adapta decisiones con estadísticas de ejecución. Column pruning evita leer columnas innecesarias cuando es posible; predicate pushdown traslada filtros al origen compatible. Ejemplo: leer → filtrar → agregación parcial → Exchange → agregación final. No confundir operadores del plan con stages observados.

**Antes de ejecutar:** usa S13 para añadir leer → filtrar → seleccionar → agrupar → show. No se garantiza ausencia de toda lectura al construir el plan (puede haber inferencia de esquema o metadatos); el cálculo de resultados es diferido. F.col hace referencia a una columna; filter conserva filas, withColumn crea o reemplaza una columna, substring extrae el año. Espera 29 de 30 ventanas con umbral 2.800.

In [ ]:
# 4 · Define un plan y ejecuta dos acciones explícitas
plan = df.filter(F.col("procesos_unicos") >= 2800).withColumn("anio", F.substring("mes", 1, 4))
print("PLAN construido; columnas:", plan.columns)
print("ACCIÓN count:", plan.count())
plan.select("anio", "mes", "cobertura").orderBy("mes").show(6, truncate=False)

**Cómo se lee:** `count` representa ventanas que cumplieron el criterio. **Qué nos dice:** cambia el universo a partir del filtro. **Qué NO permite concluir todavía:** qué porcentaje del país quedó representado. **Qué error común:** confundir el umbral de procesos únicos (2.800) con el umbral de cobertura.

## 6 · No promediar porcentajes sin revisar el denominador
Si una ventana tiene `10/100=10 %` y otra `9/10=90 %`, el promedio simple de porcentajes es 50 %, pero la cobertura combinada es `(10+9)/(100+10)=17,27 %`. Para comparar años de la misma selección, calculamos **SUM(emparejados) / SUM(procesos_unicos)** y conservamos el número de ventanas.

**Mini ficha `groupBy().agg()`:** agrupa claves; recibe agregaciones (sum/count); devuelve un nuevo plan que puede requerir shuffle. **Error común:** calcular `avg(cobertura)` y presentarlo como cobertura global.

**Cálculo manual:** enero: 201/3000; febrero: 120/2945. La cobertura conjunta es (201+120)/(3000+2945), aproximadamente 5,40 %. No es un porcentaje nacional. `groupBy` reúne numeradores y denominadores antes de dividir.

**Manual:** 10 de 100 y 9 de 10 → promedio simple 50 %, razón conjunta 19/110 = 17,27 %. `groupBy` organiza por clave; `agg` calcula sumas y conteos; `alias` nombra columnas. El resultado anual esperado es 2024: 35.048/2.015 procesos/emparejados, 2025: 35.131/1.549, 2026: 17.383/981. Los porcentajes aproximados son 5,75 %, 4,41 % y 5,64 %. Seis ventanas en 2026, no año completo.

In [ ]:
# 5 · Agregado anual ponderado: la medida tiene un denominador explícito
resumen = (df.withColumn("anio", F.substring("mes", 1, 4))
    .groupBy("anio")
    .agg(F.count("*").alias("ventanas"),
         F.sum("procesos_unicos").alias("procesos_unicos"),
         F.sum("emparejados").alias("emparejados"))
    .withColumn("cobertura_ponderada", F.col("emparejados") / F.col("procesos_unicos"))
    .orderBy("anio"))
resumen.show(truncate=False)
assert resumen.agg(F.sum("ventanas")).first()[0] == 30

**Cómo se lee:** cada fila es un año; `ventanas` revela que la cobertura temporal de los años puede ser desigual. **Qué nos dice:** razón de emparejamiento de los registros seleccionados. **Qué NO permite concluir todavía:** que 2026 sea comparable con un año completo, o que existan hechos causales. **Qué error común:** comparar años con cantidades distintas de ventanas sin observar el denominador.

## 7 · Spark SQL: dos sintaxis, mismo motor
Una **vista temporal** es un nombre consultable hasta que termine la sesión, no una tabla persistida. `spark.sql(query)` construye un DataFrame y puede ejecutar las mismas operaciones que PySpark. Validaremos el resultado con un **assert**; no basta mirar tablas parecidas.

**Mini fichas:** `createOrReplaceTempView(nombre)` registra una vista temporal; `spark.sql(SQL)` devuelve un DataFrame. Error: creer que SQL no requiere shuffle.

**Ficha:** createOrReplaceTempView registra una vista de sesión, no una tabla persistente. spark.sql devuelve un DataFrame; la acción show/collect solicita resultados. Aquí collect devuelve tres filas, no millones. La paridad se verifica con nueve decimales. Si falla, inspecciona el denominador y el agrupamiento, no cambies datos para que coincidan.

In [ ]:
# 6 · Equivalencia explícita PySpark ↔ Spark SQL
df.createOrReplaceTempView("s14_ventanas")
sql = spark.sql('''
SELECT SUBSTRING(mes,1,4) anio, COUNT(*) ventanas,
SUM(procesos_unicos) procesos_unicos, SUM(emparejados) emparejados,
SUM(emparejados)/SUM(procesos_unicos) cobertura_ponderada
FROM s14_ventanas GROUP BY SUBSTRING(mes,1,4) ORDER BY anio
''')
a = [(r.anio,int(r.ventanas),round(float(r.cobertura_ponderada),9)) for r in resumen.collect()]
b = [(r.anio,int(r.ventanas),round(float(r.cobertura_ponderada),9)) for r in sql.collect()]
assert a == b, (a,b)
print("Paridad PySpark/SQL: OK", a)

**Cómo se lee:** el mensaje OK comprueba año, número de ventanas y razón con nueve decimales. **Qué nos dice:** API y SQL representan el mismo cálculo sobre este snapshot. **Qué NO permite concluir todavía:** latencia idéntica, ni calidad del dato nacional. **Qué error común:** usar `collect()` sobre millones de filas; aquí se limita a tres filas anuales.

## 8 · Shuffle, particiones y plan de ejecución
Un **shuffle** mueve registros para que las mismas claves se combinen. El plan de `groupBy` puede incluir `Exchange`, mientras un filtro sencillo normalmente no requiere el mismo intercambio. `explain(mode="formatted")` expone el **plan físico**, que no sustituye un Query Profile real.

**Mini ficha `repartition(n, columna)`:** redistribuye registros, normalmente con un intercambio. No crea nuevos servidores. **Error común:** creer que `repartition(1)` siempre optimiza.

**Recorrido físico:** Filter aplica una condición; Exchange señala movimiento de datos; HashAggregate agrega por clave; Sort ordena cuando es necesario. Catalyst optimiza el plan lógico; AQE (Adaptive Query Execution, ejecución adaptativa) puede ajustar el plan durante la ejecución. No uses `collect()` para recuperar una tabla masiva: limita la muestra o escribe el resultado.

**Skew**: una clave concentra filas y ralentiza la tarea más pesada. **Spill**: datos intermedios salen de RAM a disco. **Broadcast**: una dimensión pequeña se distribuye a executors para evitar redistribuir la tabla grande cuando procede. **Cache**: conserva resultados reutilizables, con coste de memoria; no acelera por obligación. En serverless no se exige cache: su soporte depende de versión y preview. Examina `explain("formatted")` y compara con métricas de ejecución, no asumas que una operación siempre tiene el mismo plan.

### Del código al plan (S12–13)
Plan lógico describe qué calcular; Catalyst puede eliminar columnas y adelantar filtros; el plan físico especifica operadores como Scan, Filter, Aggregate y Exchange. AQE usa estadísticas de ejecución y puede cambiar una estrategia de unión o consolidar particiones. Una acción puede lanzar uno o varios jobs; un stage agrupa tareas de una fase y un shuffle suele separar fases. El número de etapas se observa, no se predice a partir del dibujo. Spill = intermedios temporales a disco; skew = claves/particiones desbalanceadas; broadcast = copia de una relación pequeña para unir sin redistribuir la grande por esa unión. [Spark SQL tuning](https://spark.apache.org/docs/4.0.1/sql-performance-tuning.html).

In [ ]:
# 7 · Ver planes con señales de redistribución
print("FILTRO:")
df.filter(F.col("emparejados") > 100).explain(mode="formatted")
print("\nAGRUPACIÓN:")
resumen.explain(mode="formatted")
print("Busca Exchange y Aggregate; su forma exacta depende de la versión.")
print("\nPLAN LÓGICO / OPTIMIZADO / FÍSICO:")
resumen.explain(mode="extended")

### 8B · Cruzar datos con una dimensión pequeña
Una dimensión relaciona una clave con una descripción. `join` combina tablas por clave; `left` conserva filas de la izquierda. `broadcast` indica que la dimensión es pequeña y candidata a replicarse para reducir movimiento. Nunca lo fuerces con una dimensión grande. Practica sobre 4 filas sintéticas.

**Compara estrategias, no tiempos:** primero una unión con sugerencia merge y luego broadcast explícito. Spark puede elegir otra estrategia; revisa los operadores realmente impresos. Ambas deben conservar P4 con NULL y cuatro filas. Después inspecciona 100 filas sintéticas: clave A=70, B/C/D=10. Esto muestra skew potencial por clave, no un reparto físico garantizado ni latencias. No fuerces un spill ni una falla para esta práctica.

In [ ]:
# 8B · Join de ejemplo con dimensión pequeña
procesos_ejemplo = spark.createDataFrame(
    [("P1", 0), ("P2", 1), ("P3", 0), ("P4", 2)],
    "id string, codigo int")
dimension_ejemplo = spark.createDataFrame(
    [(0, "Bien"), (1, "Servicio")], "codigo int, categoria string")
cruce = procesos_ejemplo.join(F.broadcast(dimension_ejemplo), "codigo", "left")
cruce.orderBy("id").show()
cruce.explain(mode="formatted")
assert cruce.count() == 4
assert cruce.filter(F.col("categoria").isNull()).count() == 1
print("Plan con sugerencia MERGE (no garantía de elección):")
cruce_merge = procesos_ejemplo.hint("merge").join(dimension_ejemplo.hint("merge"),"codigo","left")
cruce_merge.explain(mode="formatted")
assert cruce.exceptAll(cruce_merge).count()==0 and cruce_merge.exceptAll(cruce).count()==0
print("Claves desbalanceadas: 100 filas sintéticas; no son tiempos ni particiones físicas.")
skew_demo=spark.range(100).withColumn("clave",F.when(F.col("id")<70,"A").when(F.col("id")<80,"B").when(F.col("id")<90,"C").otherwise("D"))
skew_rows={r["clave"]:r["count"] for r in skew_demo.groupBy("clave").count().collect()}
assert skew_rows=={"A":70,"B":10,"C":10,"D":10}
print(skew_rows)

**Cómo se lee:** P4 conserva su ID y queda con categoría NULL porque no hay correspondencia. **Qué nos dice:** un left join puede preservar las filas y evidenciar una clave sin catálogo. **Qué NO permite concluir:** que un join grande conserve siempre el número de filas (una clave repetida en la dimensión puede multiplicarlas). **Qué error común:** usar inner join y perder registros silenciosamente.

## 9 · Control independiente: Spark no valida su propia respuesta
Una prueba útil compara contra un cálculo simple en Python puro, sobre las mismas 30 filas. No instalamos Dask ni ejecutamos benchmarks costosos: **corrección primero, velocidad después**. El experimento es demasiado pequeño para demostrar que Spark es más rápido.

In [ ]:
# 9 · Oráculo externo a Spark: Python puro
from collections import defaultdict
control = defaultdict(lambda: [0,0,0])
for mes,unicos,emparejados,cobertura,departamento in FILAS:
    item = control[mes[:4]]
    item[0] += 1
    item[1] += unicos
    item[2] += emparejados
referencia = [(anio, v[0], round(v[2]/v[1],9)) for anio,v in sorted(control.items())]
assert a == referencia, (a,referencia)
print("Paridad Spark/SQL/Python: OK")

**Cómo se lee:** el último OK significa que coinciden resultados anuales bajo la misma fórmula. **Qué nos dice:** es correcta esta implementación sobre el snapshot. **Qué NO permite concluir todavía:** que los datos originales sean exhaustivos o que Spark tenga ventaja de rendimiento. **Qué error común:** presentar tiempos de construir planes frente a tiempos de ejecutar acciones como comparación justa.

## 10 · Reto de transferencia: cambia una decisión
Elige un umbral entre 4 %, 7 % y 10 %: modifica **solo** `INDICE_UMBRAL` y ejecuta. Las filas representan ventanas cuya cobertura de emparejamiento dentro de la extracción supera el valor elegido. **No** se trata de una clasificación de fraude. La pregunta depende de tu propia ejecución, no de redactar una respuesta genérica.

In [ ]:
# 10 · CAMBIA ESTE ÚNICO NÚMERO: 0, 1 o 2
INDICE_UMBRAL = 1
UMBRALES = (0.04, 0.07, 0.10)
assert INDICE_UMBRAL in (0,1,2)
umbral = UMBRALES[INDICE_UMBRAL]
elegidos = df.filter(F.col("cobertura") >= umbral)
print("Umbral:", umbral, "| Ventanas:", elegidos.count())
elegidos.select("mes","cobertura").orderBy("mes").show(8, truncate=False)

**Cómo se lee:** el conteo cambia si cambias el umbral; `show(8)` presenta solo una vista. **Qué nos dice:** qué ventanas cumplen tu regla dentro del snapshot. **Qué NO permite concluir todavía:** que las ventanas sean más riesgosas; falta etiqueta o verificación independiente. **Qué error común:** usar un umbral sin documentarlo y comparar resultados de reglas distintas.

### 8A · Observabilidad: ¿qué hizo realmente Spark?
Spark UI clásico: **Jobs** trabajos y duración; **Stages** tasks, input, shuffle read/write, spill y fallos; **SQL** operadores y métricas; **Executors** recursos y tareas; **Storage** caché; **Environment** configuración. El DAG de la UI representa dependencias de ejecución; el plan físico representa operadores. Una tarea lenta puede revelar skew, pero requiere comparar distribución de tiempos y bytes. [Capturas oficiales de Spark UI](https://spark.apache.org/docs/4.0.1/web-ui.html).
En local clásico, la UI normalmente escucha en 4040 (o siguiente puerto); Codespaces permite reenviar el puerto de forma privada. Colab puede no exponer el puerto al navegador; la REST API local ofrece métricas reales sin publicar túneles. Databricks Free/serverless no tiene Spark UI clásica: ejecuta la consulta, abre See performance / Query Profile y revisa operadores, duración y filas. [Limitaciones serverless](https://docs.databricks.com/aws/en/compute/serverless/limitations) · [Query Profile](https://docs.databricks.com/aws/en/sql/user/queries/query-profile).

### Guía de interpretación de pantallas
![Jobs activos y completados, con duración y progreso](https://spark.apache.org/docs/4.0.1/img/AllJobsPageDetail3.png)
**Fuente:** [Spark 4.0.1 Web UI](https://spark.apache.org/docs/4.0.1/web-ui.html); captura oficial de ejemplo, no nuestra ejecución. Abre Job ID → stages → tasks; busca input, shuffle read/write, spill y duración. No supongas que los números de esa captura son tus métricas.

![DAG de dos etapas y Exchange](https://spark.apache.org/docs/4.0.1/img/JobPageDetail2.png)
**Cómo se lee:** Exchange conecta etapas; los nodos representan operaciones, no máquinas. **Qué nos dice:** dónde puede necesitar comunicación. **Qué NO permite concluir:** tiempos o número de servidores a partir de un dibujo. **Qué error común:** identificar cada operador como un job. Ejecuta 8A, revisa 10 categorías de 10.000. En Databricks, abre See performance bajo la celda → ejecución → See query profile. Usa las métricas que realmente estén disponibles; si faltan, informa la limitación.

**Guía de lectura S14:** Jobs → Job ID → Stages → Tasks. Duration indica duración de fase; Input y Shuffle read/write indican volumen leído/intercambiado; Spill indica estructuras temporales escritas a disco. Executors muestra procesos, memoria y CPU, no particiones. Query Profile en Serverless muestra operadores y métricas disponibles, no un inventario garantizado de máquinas. Capturas oficiales son ejemplos y no evidencias propias.

In [ ]:
# 8A · Consulta observable, 100.000 filas sintéticas, sin descarga
from urllib.request import urlopen
import json
if "dbutils" not in globals():
    spark.sparkContext.setJobGroup("s14_observabilidad", "S14: agrupar 100000 filas")
    print("Master:", spark.sparkContext.master, "| UI:", spark.sparkContext.uiWebUrl)
observado = spark.range(0,100000,1,4).withColumn("categoria", F.col("id") % 10)
agrupado_ui = observado.groupBy("categoria").count().orderBy("categoria")
agrupado_ui.explain(mode="formatted")
filas_ui = agrupado_ui.collect()
assert len(filas_ui)==10 and all(r["count"]==10000 for r in filas_ui)
print("10 categorías × 10000 = 100000; resultado real conciliado")
if "dbutils" not in globals():
    try:
        raiz_ui = spark.sparkContext.uiWebUrl.rstrip("/")
        app_id = spark.sparkContext.applicationId
        with urlopen(raiz_ui + "/api/v1/applications/" + app_id + "/jobs", timeout=5) as f:
            jobs_ui = json.load(f)
        propios = [j for j in jobs_ui if j.get("jobGroup")=="s14_observabilidad"]
        for j in propios[:8]:
            print({k:j.get(k) for k in ("jobId","status","stageIds","numTasks","numCompletedTasks")})
        if not propios:
            print("Sin jobs etiquetados en REST: abre Jobs/SQL en la UI si está accesible.")
    except Exception as e:
        print("UI REST no accesible:", type(e).__name__, "| consulta el plan y la documentación.")
    finally:
        spark.sparkContext.setLocalProperty("spark.jobGroup.id", None)
        spark.sparkContext.setLocalProperty("spark.job.description", None)
        spark.sparkContext.setLocalProperty("spark.job.interruptOnCancel", None)
else:
    print("Serverless: abre See performance / Query Profile de esta ejecución.")

**Cómo se lee:** diez claves, 10.000 registros por clave; los IDs y el número de jobs/stages son observados, no valores prefijados. **Qué nos dice:** el plan y la UI/REST relacionan código y ejecución. **Qué NO permite concluir:** número de servidores, aceleración o métricas que no se pudieron consultar. **Qué error común:** publicar el puerto de UI en Internet o buscar SparkContext clásico en serverless. Referencias: [REST](https://spark.apache.org/docs/4.0.1/monitoring.html).

## 10A · Del almacenamiento a la arquitectura
**OLTP** (transaccional): operaciones frecuentes sobre registros individuales; **OLAP** (analítico): agregaciones y exploración histórica. Un **Data Warehouse** integra datos estructurados para indicadores SQL. Un **Data Lake** conserva archivos variados (CSV, JSON, PDF, audio, Parquet) y separa almacenamiento del cómputo; requiere reglas, catálogo y acceso para no convertirse en un conjunto de archivos desordenado. **Lakehouse** combina almacenamiento abierto con gestión de tablas, transacciones, gobierno y consulta analítica. No son tres marcas ni tres motores. Schema-on-read interpreta datos al leer; schema-on-write aplica reglas al escribir. Spark procesa; Parquet guarda columnas; Delta Lake aporta transacciones; Unity Catalog administra activos; Databricks integra estas capacidades. Medallion es un patrón opcional, no requisito de Spark.

| Necesidad de Compras Claras | Componente |
|---|---|
| Guardar JSON, PDF, audios y tablas de origen | Data Lake / almacenamiento |
| Consultar indicadores SQL | Warehouse / capa analítica |
| Reutilizar fuentes y tablas controladas para ML | Lakehouse |
| Transformar sobre particiones | Spark |
| Conservar versiones de tablas | Delta Lake |
| Organizar permisos y nombres | Unity Catalog |

[Arquitectura medallion](https://docs.databricks.com/aws/en/lakehouse/medallion).

**S15–16:** imagina una API que entrega JSON, un PDF y un audio; conservarlos juntos es una necesidad de Data Lake. El almacén analítico organiza indicadores; Lakehouse agrega gestión de tablas y gobierno sobre almacenamiento abierto. No es un formato de archivo ni un motor. Spark transforma; Parquet almacena columnas; Delta incorpora log; Unity Catalog asigna nombres y permisos. OLTP registra transacciones, OLAP responde preguntas históricas. Schema-on-write y schema-on-read son decisiones de cuándo aplicar estructura.

**S15–16:** Warehouse responde preguntas analíticas históricas, Lake conserva fuentes variadas, Lakehouse combina archivos con tablas gobernadas. El catálogo registra nombres, metadatos, ubicación y permisos, no ejecuta cálculos. El SQL Warehouse de Databricks es cómputo para consultas; no confundas con el almacén lógico. Photon es un motor vectorizado nativo para operaciones compatibles SQL/DataFrames; no reemplaza toda la API Spark.

## 11 · Databricks en acción (S19–23)
Ya preparaste el entorno en bloque 1. En Databricks, **Workspace** contiene el notebook, **Catalog** muestra las tablas y sus permisos, **SQL Editor** permite consultar las tablas y **Query Profile** explica consultas. Un Volume contiene archivos, no filas SQL. Las tablas administradas tienen almacenamiento gestionado; las externas registran una ubicación de almacenamiento externa.

**Consulta real:** ejecuta 8A; debajo de la celda pulsa See performance → consulta → See query profile. Revisa operadores, filas y duración. Los menús pueden variar; si el perfil no está disponible, registra esa limitación. No busques SparkContext ni RDD clásicos en Serverless.

![Resumen oficial de Query Profile](https://docs.databricks.com/aws/en/assets/images/query-profile-summary-4e219e1124680c6f61b9a3bc426072c9.png)

[Fuente oficial](https://docs.databricks.com/aws/en/compute/serverless/notebooks) · captura de documentación, no resultado propio.

**Persistencia:** ejecuta 11B/11, 12/12B y 13. Copia los nombres completos impresos y abre Catalog → catálogo → esquema → tabla → Sample data. Verifica las filas 6/4/2 y tres filas del Gold anual. En SQL Editor puedes pegar SELECT * FROM nombre_completo (reemplaza el nombre por la tabla impresa). Si no tienes un SQL Warehouse disponible, ejecuta spark.sql("SELECT * FROM ...").show() en una celda Python del mismo notebook.

**Carga manual de un CSV para E5 o para practicar:** New → Add or upload data → Create or modify a table → elige archivo, cómputo, catálogo y esquema propios → Create new table. Comprueba encabezados, tipos y Rows span multiple lines para textos con saltos. Copia el nombre completo. [Guía oficial](https://docs.databricks.com/aws/en/ingestion/create-or-modify-table). El bloque 15 crea automáticamente la tabla de 24 textos si no subiste un CSV: no necesitas salir a Colab. No usar tablas ajenas.

**Jerarquía:** catálogo agrupa esquemas; esquema agrupa tablas y otros activos; tabla organiza filas/columnas. En Catalog → Overview comprueba columnas/tipos; Sample Data muestra una muestra (no un conteo completo). Verifica con COUNT(*) en SQL Editor o spark.sql. En el taller E5 importa el CSV con cuatro columnas de texto y Rows span multiple lines; en S14 el bloque 15 construye la tabla automáticamente.

### 11B · Medallion desde archivos con errores, práctica sintética independiente
La entrada tiene seis registros: un duplicado exacto y un valor nulo. Bronze conserva las seis filas; Silver elimina duplicado y excluye el nulo según esta regla, sin inventar valor; Gold resume dos categorías. En Colab escribimos CSV/JSON de práctica y los leemos con un esquema explícito; en Databricks, si no hay ruta a Volume, usamos exactamente las mismas filas en memoria (no fingimos lectura de archivo remoto). Si quieres practicar upload, sube el CSV a una tabla y consulta con spark.table. CSV requiere encabezados, delimitador y opción multiLine si hay saltos en campos entrecomillados. JSON aquí es JSONL: un objeto por línea. `schema` define tipos; `read.csv` y `read.json` producen DataFrames; `dropDuplicates` elimina duplicados exactos si no se especifican columnas. `filter` conserva filas válidas y `groupBy().agg()` construye indicadores.

**S17–18:** ejecuta primero la inspección visual. Bronze: C02 duplicado exacto y C03 NULL. Silver: no rellenamos NULL con cero, retiramos uno de cada problema según regla didáctica. Gold: Bien 2/4000 y Servicio 2/6000. Comprueba el conteo de exclusiones. La fuente original sigue en Bronze. En el taller Bronze recibe una tabla ya curada de 810 registros; no se espera perder filas. `exceptAll` verifica paridad incluso con duplicados; `dropDuplicates` elimina iguales; `isNull` detecta nulos; `sum` agrega.

In [ ]:
# 11B · Lectura, limpieza y conciliación de 6 registros SINTÉTICOS
import csv, json, tempfile
from pathlib import Path
filas_med = [("C01","Bien",1000),("C02","Servicio",2000),("C02","Servicio",2000),
             ("C03","Bien",None),("C04","Bien",3000),("C05","Servicio",4000)]
schema_med = "id string, categoria string, valor long"
if "dbutils" not in globals():
    carpeta_med = Path(tempfile.mkdtemp(prefix="s14_medallion_"))
    with (carpeta_med/"compras.csv").open("w", encoding="utf-8", newline="") as f:
        w = csv.writer(f); w.writerow(["id","categoria","valor"]); w.writerows(filas_med)
    with (carpeta_med/"compras.jsonl").open("w", encoding="utf-8") as f:
        for row in filas_med:
            f.write(json.dumps(dict(zip(["id","categoria","valor"],row)),ensure_ascii=False)+"\n")
    med_bronze = spark.read.option("header",True).option("multiLine",True).schema(schema_med).csv(str(carpeta_med/"compras.csv"))
    med_json = spark.read.schema(schema_med).json(str(carpeta_med/"compras.jsonl"))
    assert med_bronze.exceptAll(med_json).count()==0 and med_json.exceptAll(med_bronze).count()==0
    print("CSV y JSONL: paridad real")
else:
    med_bronze = spark.createDataFrame(filas_med,schema_med)
    print("Serverless: entrada sintética en memoria, no lectura CSV/JSON remota")
sin_duplicados = med_bronze.dropDuplicates()
rechazos_med = sin_duplicados.filter(F.col("valor").isNull())
med_silver = sin_duplicados.filter(F.col("valor").isNotNull())
med_gold = med_silver.groupBy("categoria").agg(F.count("*").alias("procesos"),F.sum("valor").alias("valor_total"))
conteos_med = [med_bronze.count(), med_silver.count(), med_gold.count()]
assert conteos_med == [6,4,2]
assert med_bronze.count()-sin_duplicados.count()==1 and rechazos_med.count()==1
assert {r["categoria"]:r["valor_total"] for r in med_gold.collect()}=={"Bien":4000,"Servicio":6000}
med_bronze.show(); med_silver.show(); med_gold.show()
print("Medallion sintético:", conteos_med)

**Cómo se lee:** 6/4/2 son filas de cada capa; 1 duplicado exacto y 1 nulo quedan auditables en Bronze, y el nulo en rechazos. **Qué nos dice:** Gold agrega valores de Silver (Bien 4000; Servicio 6000); podemos reconstruir el procesamiento desde Bronze. **Qué NO permite concluir:** que esta regla de nulos sea válida para todo SECOP o que Gold deba ser siempre agregado. **Qué error común:** eliminar Bronze o sumar dos veces un duplicado. En E5 la entrada ya viene curada, y la conciliación esperada es diferente.

## 12 · Databricks en acción: de Bronze a Silver y Gold
En una sola clase aprenderemos **PySpark y Databricks**. Creamos tres capas con el **mismo snapshot SECOP**; no abrimos un segundo curso ni repetimos las 30 ventanas desde la API:

- **Bronze (bronce):** ventana agregada tal como llega a este laboratorio, con procedencia conservada.
- **Silver (plata):** reglas explícitas de validez: positivos los denominadores, emparejados en el intervalo [0, procesos únicos], y cobertura entre 0 y 1.
- **Gold (oro):** tabla anual por año y cobertura ponderada lista para consulta. Esta es la salida que guardaremos.
- **Parquet:** archivo columnar eficiente; por sí solo no ofrece historial de transacciones de una tabla.
- **Delta Lake:** formato de tabla que usa archivos de datos y registro transaccional; permite versiones y operaciones consistentes.

**Importante:** esta es una demostración docente sobre **30 ventanas seleccionadas**. Un resultado incorrecto en Silver genera una excepción; nunca se «reparan» datos inventando valores. Los contratos de origen no están en el cuaderno.

**Mini fichas:** `filter()` descarta filas según una regla; `agg()` resume grupos; `saveAsTable()` persiste una tabla gestionada. Peligro: sobrescribir por accidente una tabla compartida. Por eso la tabla de práctica usa un sufijo aleatorio individual y no modifica bases existentes del curso.

In [ ]:
# 11 · Capas reales en Spark, con controles de grano y trazabilidad
bronze = df
invalidas = bronze.filter(
    (F.col("procesos_unicos") <= 0)
    | (F.col("emparejados") < 0)
    | (F.col("emparejados") > F.col("procesos_unicos"))
    | (F.col("cobertura") < 0)
    | (F.col("cobertura") > 1)
).count()
assert invalidas == 0, "La fuente no cumple el contrato de Silver."
silver = bronze.withColumn("anio", F.substring("mes", 1, 4))
gold = resumen
n_bronze, n_silver, n_gold = bronze.count(), silver.count(), gold.count()
assert (n_bronze, n_silver, n_gold) == (30, 30, len(a))
print("Bronze:", n_bronze, "ventanas | Silver:", n_silver,
      "ventanas | Gold:", n_gold, "años")
gold.select("anio", "ventanas", "cobertura_ponderada").show(truncate=False)

### Cómo se interpreta el pipeline
**Cómo se lee:** Bronze conserva las 30 ventanas de la extracción; Silver conserva las válidas; Gold produce una fila por año. **Qué nos dice:** las reglas pasan en esta instantánea y el indicador conserva su denominador. **Qué NO permite concluir todavía:** que se hayan procesado contratos individuales o que toda la fuente SECOP sea válida. **Qué error común:** llamar «Bronze raw» a la fuente sin aclarar que en este ejercicio es un **snapshot previamente agregado**.

**Reto de transferencia controlado:** modifica una sola regla de calidad en la celda 11 para que falle el `assert`; comprueba que el fallo detiene la salida en lugar de inventar datos. Vuelve al valor original para continuar. Esta prueba no hace llamadas a ningún servicio externo.

## 13 · Persistir el resultado: Parquet en Colab, Delta en Databricks
**Ruta Databricks (real):** el cuaderno detecta `dbutils`, obtiene el catálogo y esquema activos con `current_catalog()` / `current_schema()` y crea **una tabla de práctica con identificador aleatorio**, sin sobrescribir tablas ajenas. Dos pequeñas escrituras simulan la publicación inicial y una versión validada. Después consultaremos `DESCRIBE HISTORY` y la versión anterior.

**Ruta Colab (respaldo):** escribe el mismo Gold como **Parquet en un directorio temporal**. Se puede leer y contar, pero no obtiene historial Delta porque no hemos instalado Delta Lake ni Unity Catalog. Esto es una diferencia conceptual importante.

**Si falla crear tabla en Databricks:** se informa explícitamente el tipo de error y se conserva el resultado en memoria para terminar la clase. No se simula un historial que no existe. No se necesita token, secreto, Volume ni clúster clásico.

**Mini fichas:** `current_catalog()` y `current_schema()` informan el espacio activo; `saveAsTable()` persiste en el catálogo; `DESCRIBE HISTORY` enseña versiones de una tabla Delta. Error típico: usar `workspace.default` como una constante obligatoria para todos los estudiantes.

### Por qué Delta Lake no es Parquet
Parquet es columnar; Delta organiza archivos de datos con un registro transaccional `_delta_log`. ACID: atomicidad (todo o nada), consistencia (restricciones), aislamiento (concurrencia) y durabilidad (confirmación persistente). Una tabla puede tener versiones; time travel depende de la retención de archivos/logs, no reemplaza backups. Append agrega filas; overwrite reemplaza contenido; MERGE actualiza/inserta por clave cuando procede. En el laboratorio persistimos las tres capas sintéticas, luego el Gold real de ventanas para enseñar dos escrituras e historial. No sobrescribimos tablas compartidas.

**S21:** pulsa agregar y corregir para pasar de total 3.000 a 6.000 a 6.500. Lee versión anterior sin modificar la actual. El bloque 12 usa dos escrituras de Gold anual (inicial y validada) y el 13 consulta historial real: no esperes los tres valores ilustrativos del simulador. `saveAsTable` persiste; `spark.table` lee; `DESCRIBE HISTORY` muestra operaciones y versiones; `versionAsOf` lee una versión retenida. Delta Lake no reemplaza backups. `overwrite` solo en las tablas individuales de práctica.

**S21:** ejemplo conceptual v0 agrega A.parquet; v1 agrega B.parquet; v2 retira lógicamente A y agrega C.parquet. v0 lee A si aún existe; v2 lee B+C. El log registra cambios, no una copia completa de todos los datos por versión. El laboratorio real escribe dos versiones de Gold; no esperes el mismo contenido del ejemplo.

In [ ]:
# 12 · Una escritura pequeña, con ruta de respaldo y sin tablas compartidas
import tempfile, uuid
EN_DATABRICKS = "dbutils" in globals()
FORMATO_FINAL = "no_persistido"
TABLA_S14 = globals().get("TABLA_S14", None)
RUTA_PARQUET_S14 = globals().get("RUTA_PARQUET_S14", None)
VERSIONES_DELTA = []
if EN_DATABRICKS:
    try:
        catalogo = spark.sql("SELECT current_catalog()").first()[0]
        esquema = spark.sql("SELECT current_schema()").first()[0]
        if TABLA_S14 is None:
            TABLA_S14 = f"{catalogo}.{esquema}.s14_compras_claras_{uuid.uuid4().hex[:10]}"
        # Solo se usa una tabla individual de práctica. Sin alterar recursos del curso.
        gold.withColumn("revision", F.lit("inicial")).write.format("delta").mode(
            "overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_S14)
        gold.withColumn("revision", F.lit("validada")).write.format("delta").mode(
            "overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_S14)
        n_lectura = spark.read.table(TABLA_S14).count()
        assert n_lectura == n_gold
        FORMATO_FINAL = "delta"
        print("DELTA REAL:", TABLA_S14, "| filas:", n_lectura)
    except Exception as error:
        FORMATO_FINAL = "memoria"
        print("DELTA NO DISPONIBLE:", type(error).__name__,
              "| conserva Gold en memoria y consulta permisos del catálogo.")
else:
    if RUTA_PARQUET_S14 is None:
        RUTA_PARQUET_S14 = tempfile.mkdtemp(prefix="s14_gold_parquet_")
    gold.write.mode("overwrite").parquet(RUTA_PARQUET_S14)
    n_lectura = spark.read.parquet(RUTA_PARQUET_S14).count()
    assert n_lectura == n_gold
    FORMATO_FINAL = "parquet"
    print("PARQUET LOCAL:", RUTA_PARQUET_S14, "| filas:", n_lectura)
print("Motor:", spark.version, "| Persistencia:", FORMATO_FINAL)

### 12B · Las tres capas como activos persistentes
Ejecuta 11B primero. En Databricks usamos nombres únicos dentro del catálogo/esquema actual; en Colab guardamos tres carpetas Parquet. No es lo mismo que una tabla transaccional. La práctica conserva Bronze; los rechazos se pueden reconstruir con las reglas anteriores.

In [ ]:
# 12B · Persistencia de Bronze, Silver y Gold de práctica
import tempfile, uuid
capas_med = {"bronze":med_bronze, "silver":med_silver, "gold":med_gold}
persistidas_med = {}
tablas_med = {}
if "dbutils" in globals():
    catalogo_med = spark.sql("SELECT current_catalog()").first()[0]
    esquema_med = spark.sql("SELECT current_schema()").first()[0]
    sufijo_med = globals().get("sufijo_med",uuid.uuid4().hex[:10])
    for capa, datos in capas_med.items():
        nombre = ".".join("`"+x.replace("`","``")+"`" for x in (catalogo_med,esquema_med,"s14_"+capa+"_"+sufijo_med))
        datos.write.format("delta").mode("overwrite").saveAsTable(nombre)
        tablas_med[capa] = nombre
        persistidas_med[capa] = spark.table(nombre).count()
        print(capa,nombre,persistidas_med[capa])
    print("DELTA MEDALLION REAL")
    print("Abre Catalog y consulta Gold con: SELECT * FROM", tablas_med["gold"])
    spark.sql("SELECT * FROM "+tablas_med["gold"]+" ORDER BY categoria").show()
else:
    carpeta_capas = tempfile.mkdtemp(prefix="s14_capas_parquet_")
    for capa, datos in capas_med.items():
        ruta = carpeta_capas+"/"+capa
        datos.write.mode("overwrite").parquet(ruta)
        persistidas_med[capa] = spark.read.parquet(ruta).count()
        print(capa,ruta,persistidas_med[capa])
    print("PARQUET MEDALLION LOCAL")
assert persistidas_med == {"bronze":6,"silver":4,"gold":2}

**Cómo se lee:** los tres conteos provienen de lecturas posteriores a la escritura. **Qué nos dice:** las capas son activos recuperables; en Databricks abre Catalog y previsualiza las tres tablas. **Qué NO permite concluir:** que una escritura Parquet sea Delta o que los permisos de otro usuario funcionen. **Qué error común:** continuar si la escritura falla y declarar las tablas creadas. Si falta cuota/permisos, conserva el resultado 11B y registra el error, o usa la ruta local explícita.

### Comparación conceptual, basada en la ejecución
**Cómo se lee:** `DELTA REAL` significa que Databricks pudo escribir una tabla gestionada; `PARQUET LOCAL` significa que Colab guardó un archivo columnar; `DELTA NO DISPONIBLE` significa que **no** se obtuvo persistencia administrada.

**Qué nos dice:** los tres caminos pueden comprobar el cálculo del indicador, pero sus capacidades de gobernanza son distintas. **Qué NO permite concluir todavía:** que Parquet tenga historial de transacciones o que una tabla de tres filas se beneficie del cómputo distribuido. **Qué error común:** afirmar que Databricks es únicamente «un Colab con otra pantalla» o que Delta es otro nombre de Parquet.

In [ ]:
# 13 · Historial de versiones auténtico, solo si Delta pudo escribirse
if FORMATO_FINAL == "delta":
    historial = spark.sql(f"DESCRIBE HISTORY {TABLA_S14}")
    version_rows = historial.select("version", "operation").orderBy(
        F.col("version").desc()).limit(5).collect()
    VERSIONES_DELTA = [int(r.version) for r in version_rows]
    print("Versiones presentes:", VERSIONES_DELTA)
    for r in version_rows:
        print("Versión", int(r.version), "->", r.operation)
    anterior = spark.read.option("versionAsOf", 0).table(TABLA_S14)
    print("Versión inicial:")
    anterior.select("anio", "revision").orderBy("anio").show(truncate=False)
    print("Versión actual:")
    spark.table(TABLA_S14).select("anio", "revision").orderBy("anio").show(truncate=False)
else:
    print("SIN HISTORIAL DELTA: esta ruta no creó tabla Delta.",
          "Revisa el plan conceptual y usa Parquet/memoria para cerrar la clase.")

### El historial no es una copia de seguridad perpetua
**Cómo se lee:** versión 0 contiene `revision=inicial`; la versión reciente contiene `revision=validada` si las escrituras se ejecutaron. **Qué nos dice:** Delta registra operaciones y permite consultar versiones retenidas. **Qué NO permite concluir todavía:** cuánto tiempo permanecerán disponibles ni que el historial sustituya los respaldos; las políticas de retención importan. **Qué error común:** confundir el historial de una tabla con copias inmutables de toda la información SECOP.

**Nota:** al volver a ejecutar la celda de escritura, la tabla individual acumula nuevas versiones. Reutiliza el mismo identificador durante la sesión; al terminar puedes eliminar solo esa tabla propia ejecutando `spark.sql(f"DROP TABLE IF EXISTS {TABLA_S14}")` (en Databricks), nunca borrando catálogos ni tablas de otros.

## 14 · Databricks Jobs: ejecutar el notebook de forma operativa
Un **Job** (trabajo programado) permite definir una tarea que ejecuta un notebook y revisar resultado, errores, duración y reintentos. No es una nueva biblioteca de Spark. **En esta clase basta crear el flujo conceptual y conocer la interfaz; no programamos ejecuciones repetidas** que puedan consumir cuotas gratuitas.

En Databricks: 1) guarda/importa este notebook en Workspace; 2) abre **Jobs & Pipelines** (la ubicación puede variar); 3) elige crear un Job de prueba con una tarea de notebook; 4) selecciona Serverless y **sin horario recurrente**; 5) revisa qué código se ejecutaría, las dependencias y cómo acceder al historial de ejecuciones. **No lo ejecutes si tu cuota está cerca del límite**.

**Recorrido comprobable:** notebook → tarea → ejecución → logs → tabla Delta. **Qué NO permite concluir todavía:** que esté automatizado o que exista un SLA en Free Edition. En el plan Free Edition hay cuotas y un límite de tareas concurrentes. Documentación: https://docs.databricks.com/aws/en/getting-started/free-edition-limitations

**Cierre único de Spark + Databricks:** ya sabemos construir la consulta, interpretarla, persistir una tabla y observar su versión; la siguiente sesión del curso se dedicará a integrar el proyecto, no a repetir Databricks.

## 15 · Evidencia automática, decisiones cerradas
El siguiente expediente guarda la fórmula, los recuentos, el entorno y la procedencia. Elige un motor con un índice de `(pandas, Dask, Spark)` para un **escenario de producción hipotético que sí requiera procesamiento distribuido**. Esta decisión no altera los hechos: 30 filas no requieren Spark.

Este expediente (13B) integra los bloques 9 y 10 con 11–13: ejecútalo después de esos bloques y antes de la sección 16.

In [ ]:
# 13B · Expediente verificable, sin credenciales ni archivos pesados
import json
MOTORES = ("pandas","Dask","Spark")
INDICE_MOTOR = 2  # 0/1/2: decisión cerrada para un escenario de crecimiento
assert INDICE_MOTOR in (0,1,2)
expediente = {
 "sesion":14,"fuente":PROCEDENCIA,"entorno":ENTORNO,"spark_version":spark.version,
 "filas":len(FILAS),"resultado":[{"anio":x,"ventanas":n,"cobertura":c} for x,n,c in a],
 "paridad":{"spark_sql":a==b,"python":a==referencia},
 "seleccion":{"umbral":umbral,"ventanas":elegidos.count()},
 "lakehouse":{"bronze":n_bronze,"silver":n_silver,"gold":n_gold,
              "formato":FORMATO_FINAL,"tabla_delta":TABLA_S14 if FORMATO_FINAL=="delta" else None,
              "versiones_delta":VERSIONES_DELTA},
 "motor_escenario_futuro":MOTORES[INDICE_MOTOR],
 "limite":"30 ventanas de extracción, NO censo de contratos ni evidencia de fraude"
}
assert all(expediente["paridad"].values())
print(json.dumps(expediente,ensure_ascii=False,indent=2))

**Cómo se lee:** verifica `fuente`, `filas`, `resultado`, `paridad`, `umbral` y `limite`. **Qué nos dice:** qué calculaste y bajo qué condiciones. **Qué NO permite concluir todavía:** escalabilidad, latencia o cobertura nacional. **Qué error común:** compartir solo el gráfico sin el denominador ni versión.

**Autoevaluación formativa (no calificable):** (1) una partición es un servidor: **falso**; (2) `count()` es una acción: **verdadero**; (3) Spark SQL usa siempre otro motor: **falso**; (4) una tabla con 30 filas demuestra velocidad distribuida: **falso**; (5) Databricks Free Edition obliga a crear un cluster clásico: **falso**; (6) Delta registra versiones y Parquet suelto no aporta por sí solo historial de transacciones: **verdadero**.

**Puente siguiente sesión:** integración del proyecto, no repetir Spark/Databricks.\n\n**Referencias:** [PySpark DataFrames](https://spark.apache.org/docs/latest/api/python/user_guide/dataframes.html) · [Databricks Free Edition](https://docs.databricks.com/aws/en/getting-started/free-edition-limitations).

## 16 · Laboratorio de transferencia: clasificar bienes y servicios con Spark MLlib
**¿Qué es MLlib?** Es la biblioteca de aprendizaje automático de Apache Spark. Su API principal basada en DataFrames es `pyspark.ml`; la API antigua basada en RDD (`pyspark.mllib`) está en mantenimiento. Incluye algoritmos de clasificación, regresión, agrupamiento, recomendación, transformaciones de características, evaluación y persistencia de modelos y pipelines.

| Problema | Entrada y salida | Ejemplos de algoritmos |
|---|---|---|
| Clasificación | Variables → categoría | LogisticRegression, RandomForestClassifier |
| Regresión | Variables → valor numérico | LinearRegression |
| Agrupamiento | Variables sin etiqueta → grupo | KMeans |
| Recomendación | Interacciones → preferencias estimadas | ALS |

**Objetivo del caso:** construir un clasificador de descripciones de compras, 0 = bien y 1 = servicio. Utilizaremos 24 textos sintéticos independientes del taller: 16 train para aprender y 8 test para evaluar. No medimos escalabilidad ni calidad real de SECOP. Las particiones contienen datos, no modelos independientes.

**Secuencia:** sigue los bloques 14–17 después de completar las operaciones anteriores de este cuaderno. No instales otro Spark en Databricks ni programes Jobs recurrentes.

**Puente al taller:** E5 usa CSV curado de 810 registros, conserva split; E6 utiliza ese split, HashingTF más amplio e IDF ajustado en train; E7 usa las predicciones sklearn y Spark aporta evidencia remota. No copiar resultados sintéticos como evidencias del taller. [Guía oficial MLlib](https://spark.apache.org/docs/4.0.1/ml-guide.html).

**Demostración en Databricks (S23):** 14 genera los textos; 15 crea y relee una tabla Delta individual (o valida exactamente el CSV de práctica que subiste); 16 ajusta y predice; 17 evalúa. No debes cargar archivos externos ni cambiar de herramienta para ejecutar esta práctica. En Catalog consulta la tabla de 24 textos. El modelo se mantiene en la sesión; no se publica un servicio de inferencia. Si hay errores de permisos o cuota, no se considera completada la persistencia Delta.

In [ ]:
# 14 · Datos didácticos autocontenidos: no descargan el dataset evaluado
import csv
from pathlib import Path
bienes = [
    "compra de computadores portátiles", "adquisición de sillas para oficinas",
    "compra de impresoras y accesorios", "suministro de mobiliario escolar",
    "adquisición de equipos de laboratorio", "compra de materiales eléctricos",
    "suministro de papel y carpetas", "adquisición de herramientas manuales",
    "compra de escritorios metálicos", "suministro de equipos informáticos",
    "adquisición de libros educativos", "compra de repuestos de vehículos"
]
servicios = [
    "prestación de servicios de mantenimiento", "servicio de vigilancia de edificios",
    "prestación de servicios de capacitación", "servicio de soporte técnico",
    "prestación de servicios de asesoría", "servicio de transporte escolar",
    "servicio de limpieza de oficinas", "prestación de servicios de consultoría",
    "servicio de mantenimiento eléctrico", "prestación de servicios de formación",
    "servicio de reparación de equipos", "servicio de logística de eventos"
]
filas_demo = [(f"D{etiqueta}-{i:02d}", texto, float(etiqueta),
               "train" if i < 8 else "test")
              for etiqueta, grupo in enumerate((bienes, servicios))
              for i, texto in enumerate(grupo)]
schema_demo = "id_demo string, texto string, label double, split string"
demo = spark.createDataFrame(filas_demo, schema_demo)
demo.groupBy("split", "label").count().orderBy("split", "label").show()
assert demo.count() == 24
# En Colab: exporta un CSV para practicar la carga manual en Databricks.
if "dbutils" not in globals():
    ruta_demo = Path("s14_demo_bienes_servicios.csv")
    with ruta_demo.open("w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id_demo", "texto", "label", "split"])
        w.writerows(filas_demo)
    print("CSV de práctica:", ruta_demo.resolve())

**Cómo se lee:** cada grupo combina split y etiqueta; deben verse cuatro grupos, ocho ejemplos train y cuatro test por clase. **Qué nos dice:** tenemos separación explícita y clases representadas. **Qué NO permite concluir:** que los textos sintéticos sean representativos de contratos reales. **Qué error común:** reutilizar test para aprender el vocabulario o cambiar la división al comparar motores.

**Databricks en la práctica:** desde Colab descarga s14_demo_bienes_servicios.csv en Archivos. En Databricks Free, Catalog → Add/Upload data → Create table from upload; confirma encabezados y tipos. Crea una tabla Delta propia con nombre diferente al taller. Copia su nombre completo catálogo.esquema.tabla. En la siguiente celda modifica TABLA_DEMO solo si importaste el CSV. Si no hay acceso, usa el DataFrame autocontenido. Fuente: [carga de archivos](https://docs.databricks.com/aws/en/ingestion/create-or-modify-table) · [Spark ML serverless](https://docs.databricks.com/aws/en/release-notes/serverless/environment-version/four).

In [ ]:
# 15 · Persistir y releer los 24 textos antes de entrenar en Databricks
import uuid
TABLA_DEMO = ""  # Si subiste el CSV de práctica, escribe TU nombre completo; si no, se crea una tabla.
if "dbutils" in globals() and not TABLA_DEMO:
    cat_demo = spark.sql("SELECT current_catalog()").first()[0]
    sch_demo = spark.sql("SELECT current_schema()").first()[0]
    sufijo_ml = globals().get("sufijo_ml", uuid.uuid4().hex[:10])
    TABLA_DEMO = ".".join("`"+x.replace("`","``")+"`" for x in (cat_demo, sch_demo, "s14_textos_ml_"+sufijo_ml))
    # Tabla individual; no sobrescribe tablas compartidas.
    demo.write.format("delta").mode("overwrite").saveAsTable(TABLA_DEMO)
    print("Tabla Delta de entrenamiento creada:", TABLA_DEMO)
if TABLA_DEMO:
    entrada_demo = spark.table(TABLA_DEMO).select(
        F.col("id_demo").cast("string"), F.col("texto").cast("string"),
        F.col("label").cast("double"), F.col("split").cast("string"))
    assert entrada_demo.exceptAll(demo).count()==0 and demo.exceptAll(entrada_demo).count()==0, "La tabla debe contener exactamente los 24 textos y split de práctica"
    demo = entrada_demo
    print("Tabla de entrada releída:", TABLA_DEMO)
else:
    print("Spark local: entrada de práctica en memoria; no es una tabla Delta.")
assert demo.count() == 24
assert demo.select("id_demo").distinct().count() == 24
assert demo.filter(F.col("id_demo").isNull() | F.col("texto").isNull() | F.col("label").isNull() | F.col("split").isNull()).count() == 0
assert demo.filter(~F.col("label").isin(0.0, 1.0) | ~F.col("split").isin("train", "test")).count() == 0
train_demo = demo.filter(F.col("split") == "train").repartition(2)
test_demo = demo.filter(F.col("split") == "test")
assert train_demo.count() == 16 and test_demo.count() == 8
train_demo.groupBy(F.spark_partition_id().alias("particion")).count().show()
demo.createOrReplaceTempView("s14_demo_textos")
spark.sql("SELECT split, label, COUNT(*) AS n FROM s14_demo_textos GROUP BY split, label ORDER BY split, label").show()

### 16.1 · Cómo aprende un modelo de MLlib (S22–23)
**Intuición:** un modelo de regresión logística calcula una probabilidad a partir de características numéricas y pesos aprendidos. Durante `fit`, el algoritmo compara predicción y etiqueta, calcula una pérdida (qué tan equivocadas son las probabilidades) y ajusta los pesos para reducirla. Los cálculos parciales pueden ejecutarse sobre diferentes particiones y combinarse para ajustar un solo modelo.

#### Simulación manual de una iteración (no ejecución de MLlib)
Supongamos una característica escalar x (señal de servicio +1 o bien −1), una etiqueta y (0 o 1), peso inicial w=0, sin intercepto ni regularización. La probabilidad p=1/(1+exp(−w·x)) es 0,5 para todos los ejemplos. El gradiente de la pérdida logística de un ejemplo es (p−y)·x: indica cómo cambia la pérdida con el peso.

| Partición | Ejemplos (x,y) | Gradientes individuales | Suma | Cantidad |
|---|---|---|---:|---:|
| P1 | (1,1), (1,1) | −0,5; −0,5 | −1 | 2 |
| P2 | (−1,0), (1,0) | −0,5; +0,5 | 0 | 2 |
| Total | 4 | | −1 | 4 |

Gradiente global = −1/4 = −0,25. Con tasa de aprendizaje ilustrativa 1, el nuevo peso es w=0−1×(−0,25)=0,25. Para x=1, la nueva probabilidad de servicio es aproximadamente 56,2 %. En otra iteración se recalcularían los gradientes con el peso actualizado. Se combinan sumas y cantidades; no se promedian medias de particiones de tamaños distintos sin ponderar. El algoritmo real de regresión logística de Spark emplea sus propios optimizadores, un vector de pesos, intercepto y regularización; la simulación no reproduce literalmente su ejecución.

**Cómo se lee:** los dos parciales contribuyen a un solo peso. **Qué nos dice:** una actualización del modelo puede combinar información de las particiones. **Qué NO permite concluir:** rendimiento real, convergencia ni calidad predictiva. **Qué error común:** asumir que cada partición entrena un modelo independiente o que dos particiones equivalen a dos máquinas.

#### Del texto al modelo real: Pipeline
| Etapa | Entrada | Salida | Qué hace |
|---|---|---|---|
| Tokenizer | texto | tokens | separar palabras |
| HashingTF | tokens | tf | contar frecuencias en un vector de tamaño fijo; posibles colisiones |
| IDF | tf | features | aprender ponderaciones según frecuencia documental en train |
| LogisticRegression | features + label | modelo | aprender coeficientes para estimar la clase |

**Transformer** aplica `transform` (Tokenizer, HashingTF); **Estimator** aprende con `fit` (IDF y LogisticRegression). `Pipeline` encadena etapas; `fit(train_demo)` devuelve un PipelineModel con los parámetros aprendidos; `transform(test_demo)` aplica esos parámetros sin volver a ajustar. Un vector disperso guarda las posiciones no nulas. La representación textual es un conocimiento de sesiones anteriores; aquí se conserva el pipeline necesario para ejecutar MLlib y el taller, sin volver a desarrollar TF-IDF.

**Parámetros del código:** `numFeatures=256` define longitud del vector; `maxIter=15` limita iteraciones del optimizador; `regParam=0.1` controla penalización de complejidad; `featuresCol` y `labelCol` identifican características y etiqueta. `prediction` es la clase estimada, `probability` contiene probabilidades. La evaluación real se realiza en test en bloque 17.

**HAZ ESTO AHORA:** ejecuta 15 y comprueba en Catalog la tabla Delta de 24 textos. Si falla la creación, revisa permisos/contexto del bloque 1 y vuelve a ejecutar. Después ejecuta 16; Databricks requiere entorno serverless compatible con pyspark.ml (4 o posterior según disponibilidad); si no está disponible, registra la limitación y utiliza Colab PySpark. No instales Spark dentro de Databricks. [Pipelines MLlib](https://spark.apache.org/docs/4.0.1/ml-pipeline.html) · [Clasificación y regresión](https://spark.apache.org/docs/4.0.1/ml-classification-regression.html).

In [ ]:
# 16 · Entrenamiento y evaluación reales, sin valores de F1 prefijados
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
pipeline_demo = Pipeline(stages=[
    Tokenizer(inputCol="texto", outputCol="tokens"),
    HashingTF(inputCol="tokens", outputCol="tf", numFeatures=256),
    IDF(inputCol="tf", outputCol="features"),
    LogisticRegression(featuresCol="features", labelCol="label", maxIter=15, regParam=0.1)
])
modelo_demo = pipeline_demo.fit(train_demo)
pred_demo = modelo_demo.transform(test_demo)
pred_demo.select("id_demo", "texto", "label", "prediction", "probability").orderBy("id_demo").show(truncate=False)
# El PipelineModel aplica Tokenizer, HashingTF e IDF ya ajustado.
pred_demo.select("id_demo", "tokens", "tf", "features").limit(2).show(truncate=70)
assert pred_demo.count() == 8

**Cómo se lee:** label es la clase real; prediction es la clase predicha. En las dos filas de inspección, tokens son palabras, tf es el vector disperso de frecuencias y features es el vector ponderado. **Qué nos dice:** el PipelineModel aplica transformaciones aprendidas únicamente en train a documentos nuevos. **Qué NO permite concluir:** que generalice a SECOP ni que usar particiones acelere el cálculo. **Qué error común:** ajustar IDF con todos los documentos o leer las ocho predicciones como si fueran de entrenamiento.

### 16.2 · Evaluar sin confundir exactitud con F1
La matriz tiene filas reales y columnas predichas. Para cada clase: precisión = VP/(VP+FP), sensibilidad = VP/(VP+FN) y F1 = 2VP/(2VP+FP+FN), con 0 si el denominador es cero. F1 macro promedia ambos F1 por igual; F1 ponderado los promedia por soporte. Ejemplo manual ajeno al taller: matriz [[3,1],[2,2]]. Exactitud 5/8=0,625; F1 clase 0=6/9, clase 1=4/7, macro≈0,6190. Calcula ahora las métricas del modelo realmente ejecutado.

**S24:** la matriz ilustrativa [[3,1],[2,2]] tiene exactitud 5/8=0,625 y F1 macro≈0,6190. Cambia un acierto de clase 1 por error; la métrica baja. Esa matriz NO es la de MLlib real. Bloque 17 calcula matriz de tu ejecución, exactitud, F1 por clase y promedio; el evaluador Spark metricName=f1 es ponderado. E5/E6 usan el mismo split del proyecto y devuelven JSON; E7 usa predicciones sklearn y genera el HTML. No cambies el dataset ni las reglas de evaluación.

In [ ]:
# 17 · Matriz completa, F1 macro y F1 ponderado, sin reentrenar
matriz_demo = pred_demo.groupBy("label", "prediction").count().orderBy("label", "prediction")
matriz_demo.show()
pares = {(int(r["label"]), int(r["prediction"])): int(r["count"])
         for r in matriz_demo.collect()}
m = [[pares.get((i,j), 0) for j in (0,1)] for i in (0,1)]
assert sum(map(sum, m)) == 8
f1_clase = []
for i in (0,1):
    vp = m[i][i]
    fp = m[1-i][i]
    fn = m[i][1-i]
    den = 2*vp+fp+fn
    f1_clase.append(2*vp/den if den else 0.0)
f1_macro_demo = sum(f1_clase)/2
f1_ponderado = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1").evaluate(pred_demo)
f1_por_soporte = sum(sum(m[i])*f1_clase[i] for i in (0,1))/8
assert abs(f1_ponderado - f1_por_soporte) < 1e-8
print("Matriz filas reales, columnas predichas:", m)
print("Exactitud:", (m[0][0]+m[1][1])/8)
print("F1 por clase:", f1_clase, "| F1 macro:", round(f1_macro_demo,4),
      "| F1 ponderado:", round(f1_ponderado,4))

**Cómo se lee:** diagonal = aciertos, fuera de la diagonal = errores. En esta muestra balanceada ambos F1 coinciden; con soportes distintos pueden diferir. **Qué nos dice:** la evaluación es reproducible sobre el test fijo. **Qué NO permite concluir:** que F1 represente contratos reales ni sea evidencia de escalabilidad. **Qué error común:** escoger hiperparámetros mirando test. **Decisión formativa:** cambia solo el reparto de train a 4 particiones y observa el conteo por partición, sin reentrenar ni seleccionar con test; identifica un ID acertado y uno fallado si existe. En el taller conserva el split y calcula F1 macro con sus propios resultados.

## Hoja de consulta y cierre
| Quiero... | Uso... |
|---|---|
| Leer CSV/JSON | spark.read.schema(...).csv/json(...) |
| Consultar plan | explain("formatted") |
| Observar trabajos | Spark UI local / REST; Query Profile remoto |
| Agrupar | groupBy().agg() |
| Persistir | write.parquet / write.format("delta").saveAsTable |
| Ver versiones Delta | DESCRIBE HISTORY / versionAsOf |
| Entrenar / predecir | fit(train) / transform(test) |
| Transferir evidencias | E5/E6 JSON → carpeta salidas local |
**Para llevar:** Spark no es más rápido por definición; su valor está en la capacidad de estructurar, optimizar, distribuir y observar flujos. Lakehouse organiza datos; Delta aporta consistencia; Databricks integra la plataforma. Las métricas de este laboratorio no miden fraude ni rendimiento nacional.